In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch04. tensorflow v1</font>

# 1. tensorflow v1


| **TensorFlow 1.x 구버전** | **TensorFlow 2.x 신버전** |
| :--- | :--- |
| `tf.placeholder()` | `tf.keras.Input()` 또는 함수 인자 사용 |
| `tf.Variable()` + `Session.run()` | `tf.Variable()` (즉시 연산 가능) |
| `sess.run(train_step, feed_dict={...})` | `tf.GradientTape()` 또는 `model.fit()` |
| **정적 그래프 (Define-and-Run)**<br>그래프 설계 후 `tf.Session()`에서 실행 | **즉시 실행 (Eager Execution / Define-by-Run)**<br>Python/NumPy처럼 코드 선언 즉시 결과 계산 |
| C++ 최적화 및 분산 환경 처리 유리 | 직관적인 디버깅 및 간결한 Keras API 통합 |
| `tf.placeholder`, `tf.Session`, `Graph` | `tf.keras`, `tf.GradientTape`, `@tf.function` |

In [ ]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # tensorflow v2 비활성화 및 v1 활성화
import numpy as np
import pandas as pd

## Tensorflow
- 데이터 흐름 그래프(Tensor 흐름을 나타내는 설계도)를 사용하는 수치 계산 라이브러리
- 구성 요소 : tensor(다차원 배열 데이터 구조), node(수치 연산 및 동작), edge(흐름 및 관계)
- `sess = tf.Session()`을 이용하여 실행 환경(세션) 생성
- `sess.run()`을 통해 결과 확인

### tensor 정의
- 상수 node, 변수 node, 연산 node 정의 = tensor 객체 정의

In [ ]:
node1 = tf.constant('Hello, Tensorflow')
print(node1) # node : 설계도 정보 출력됨

### 세션 생성

In [ ]:
sess = tf.Session()

### 실행

In [ ]:
print(sess.run(node1)) # 문자열 출력의 경우, 바이너리로 출력됨
print(sess.run(node1).decode())

### 연산 node

In [ ]:
# 1. node 정의
node1 = tf.constant(10, dtype=tf.float16)
node2 = tf.constant(20, dtype=tf.float16) # 텐서의 연산에서, 각 노드의 타입은 같아야함

# 2. 연산 node 정의
node3 = tf.add(node1, node2)

# 3. 세션 생성
sess = tf.Session()

# 4. 실행
n1, n2, n3 = sess.run([node1, node2, node3])
print(n1, n2, n3)

### 타입 변경

In [ ]:
node1 = tf.constant(np.array([10, 20, 30]), dtype=tf.int16)
node2 = tf.cast(node1, dtype=tf.float32)
sess = tf.Session()
print(sess.run([node1, node2]))

### 평균값 계산
- tf.reduce_mean()

In [ ]:
data = np.array([1., 2, 3, 4]) # data의 타입이 정수일 경우, 계산 결과도 정수로 산출됨 → 정확한 결과 수집을 위해 실수 타입 설정 필요
m = tf.reduce_mean(data)
sess = tf.Session()
sess.run(m)

### 난수 발생
- tf.random_normal([shape]) : 평균 0, 표준편차 1인 shape 모양의 난수 배열 생성 (기본값 : float32)

In [ ]:
w = tf.random_normal([1, 2])
sess = tf.Session()
sess.run(w)

### 변수 node

In [ ]:
w = tf.Variable(tf.random_normal([1]))
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # 설계도(그래프)에 선언된 모든 변수들의 초기값을 일괄 주입
sess.run(w)

# 2. tf v1 회귀분석
- tensorflow v1을 이용한 회귀분석 구현

## 2.1 독립변수 x, 종속변수 y 각각 1개

### 설계도 정의

In [ ]:
# 데이터셋 확보
x = np.array([1,2,3])
y = np.array([2,3,4])

In [ ]:
# weight와 bias
w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

In [ ]:
# hat, hypothesis : numpy 배열을 출력
H = w * x + b

In [ ]:
# cost function (손실함수 mse ; H-y 제곱의 평균)
cost = tf.reduce_mean(tf.square(H-y))

In [ ]:
# optimizer를 통한 cost 최솟값 도출 시도
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)

In [ ]:
# 세션 생성
sess = tf.Session()

In [ ]:
# w와 b 초기화
sess.run(tf.global_variables_initializer())

### 학습

In [ ]:
for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b])
    if step%500==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

In [ ]:
w_, b_ = sess.run([w[0], b[0]])
w_, b_

In [ ]:
def predict(x):
    return x*w_ + b_

In [ ]:
predict(5)

## 2.2 placeholder
- placeholder : 외부 데이터(X, y)를 연산 그래프 내부로 주입하기 위한 노드

In [ ]:
X = tf.placeholder(dtype=np.float32)
H = w_*X + b_
sess = tf.Session()
sess.run([H, X], {X:2.5})

In [ ]:
sess.run(H, {X: np.array([2.5, 3, 3.5])})

In [ ]:
x_data = np.array([1,2,3])
y_data = np.array([2,3,4])

# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

H = w * x + b

cost = tf.reduce_mean(tf.square(H-y))

train = tf.train.GradientDescentOptimizer(learning_rate=0.005).minimize(cost)

sess = tf.Session()

sess.run(tf.global_variables_initializer())

for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b], feed_dict={x:x_data, y:y_data})
    if step%500==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

In [ ]:
# 예측하기
# feed_dict : 외부에서 정의한 입력값(tf.placeholder)에 실제 데이터를 주입(Feed)
sess.run(H, feed_dict={x:2.5})

In [ ]:
sess.run(H, feed_dict={x:np.array([2.5, 3.5])})

## 2.3 scale
- scale이 다른 데이터의 회귀분석

In [ ]:
# scale 조정 X
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])

x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

H = w * x + b

cost = tf.reduce_mean(tf.square(H-y))

# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.1)
# 손실값(Cost)이 발산하여 nan이 출력
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.001)
train = optimizer.minimize(cost)

sess = tf.Session()

sess.run(tf.global_variables_initializer())

for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b], feed_dict={x:x_data, y:y_data})
    if step%500==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

### scale 조정
- 모든 데이터를 일정 범위 내로 조정
    - normalization (정규화) $= \frac{X - X_{\min}}{X_{\max} - X_{\min}}$ : 데이터의 범위를 0과 1 사이로 변환
        - 라이브러리 : sklearn.preprocessing.MinMaxScaler
    - standardization (표준화) $= \frac{X - X_{\mu}}{X_{\sigma}}$ : 데이터의 평균${\mu}$이 0이고 표준편차${\sigma}$가 1인 정규분포 형태로 변환
        - 라이브러리 : sklearn.preprocessing.StandardScaler

In [ ]:
# 라이브러리를 사용하지 않은 정규화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])

norm_scaled_x_data = (x_data - x_data.min())/(x_data.max() - x_data.min())
norm_scaled_y_data = (y_data - y_data.min())/(y_data.max() - y_data.min())

In [ ]:
# 라이브러리를 활용한 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# MinMaxScaler : 2차원 배열([Samples, Features])을 입력으로 받음
x_data = np.array([1,2,5,8,10]).reshape(-1,1)
y_data = np.array([5,15,68,80,95]).reshape(-1,1)

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

norm_scaled_x_data = scaler_x.fit_transform(x_data)
norm_scaled_y_data = scaler_y.fit_transform(y_data)

np.column_stack([x_data, norm_scaled_x_data, y_data, norm_scaled_y_data])

In [ ]:
# 라이브러리를 사용하지 않은 표준화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])

std_scaled_x_data = (x_data - x_data.mean())/(x_data.std())
std_scaled_y_data = (y_data - y_data.mean())/(y_data.std())
print(np.column_stack([x_data, std_scaled_x_data]))
print()
print(np.column_stack([y_data, std_scaled_y_data]))

In [ ]:
# 라이브러리를 활용한 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# StandardScaler : 2차원 배열([Samples, Features])을 입력으로 받음
x_data = np.array([1,2,5,8,10]).reshape(-1,1)
y_data = np.array([5,15,68,80,95]).reshape(-1,1)

# 범위 혼용 방지를 위한 객체 개별 생성
scaler_x = StandardScaler()
scaler_y = StandardScaler()

std_scaled_x_data = scaler_x.fit_transform(x_data)
std_scaled_y_data = scaler_y.fit_transform(y_data)

print(np.column_stack([x_data, std_scaled_x_data]))
print()
print(np.column_stack([y_data, std_scaled_y_data]))

In [ ]:
# scale 조정 O

x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

w = tf.Variable(tf.random_normal([1]), name='weight')
b = tf.Variable(tf.random_normal([1]), name='bias')

H = w * x + b

cost = tf.reduce_mean(tf.square(H-y))

optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.001)
train = optimizer.minimize(cost)

sess = tf.Session()

sess.run(tf.global_variables_initializer())

for step in range(1, 10001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b], feed_dict={x:norm_scaled_x_data, y:norm_scaled_y_data})
    if step%3000==1:    
        print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')
print(f'{step} 번째 cost = {cost_val}, w = {w_val}, b = {b_val}')

### scale 복원
- inverse_transform()

In [ ]:
scaler_x.inverse_transform(std_scaled_x_data)

In [ ]:
scaler_y.inverse_transform(std_scaled_y_data)

## 2.4 독립변수 x 3개, 종속변수 y 1개

In [ ]:
# data set
X_data = np.array([[73, 80, 75],     # X : 학습 시 5행 3열 / 예측 시 ?행 3열
                   [93, 88, 93],
                   [89, 91, 90],
                   [96, 98, 100],
                   [73, 66, 70]])
y_data = np.array([[152],            # Y : 학습 시 5행 1열 / 예측 시 ?행 1열
                   [185], 
                   [180], 
                   [196], 
                   [142]])

X = tf.placeholder(shape=[None, 3], dtype=tf.float32) # placeholder에서 입력은 2차원
y = tf.placeholder(shape=[None, 1], # N행(가변적) 1열 입력
                   dtype=tf.float32)

# w & b
W = tf.Variable(tf.random_normal([3, 1])) # 행렬 연산을 위해 입력 값의 열 개수에 맞추어 행 개수 생성
b = tf.Variable(tf.random_normal([1]))

# H = X @ W + b
H = tf.matmul(X, W) + b

# cost
cost = tf.reduce_mean(tf.square(H - y))

# optimizer & train
train = tf.train.GradientDescentOptimizer(learning_rate=0.000045).minimize(cost) # GradientDescentOptimizer : 경사하강법

# sess & variable initialize
sess = tf.Session()
sess.run(tf.global_variables_initializer())

# learn
for step in range(1, 100001):
    _, cost_val = sess.run([train, cost], feed_dict={X:X_data, y:y_data})
    
    if step%10000==1:
        print(f'{step}번째 cost : {cost_val}')
        
print(f'{step}번째 cost : {cost_val}')

```
[현재 W, b]  ──(계산)──>  [현재 cost]  ──(미분/기울기)──>  [새로운 W, b 저장]
      ▲                                                                │
      └─────────── (다음 루프 반영) ────────────┘
```

In [ ]:
# predict
input_data = np.array([[72, 79, 74],
                       [92, 87, 92]])
hat = sess.run(H, feed_dict={X:input_data})
hat

In [ ]:
sess.run(H, feed_dict={X:[[90,85,90]]})

### 다중 독립변수의 scale 조정 및 학습

In [ ]:
from sklearn.preprocessing import StandardScaler

X_data = np.array([[73, 80, 75],  
                   [93, 88, 93],
                   [89, 91, 90],
                   [96, 98, 100],
                   [73, 66, 70]])
y_data = np.array([[152],  
                   [185], 
                   [180], 
                   [196], 
                   [142]])

scaler_x = StandardScaler()
scaler_y = StandardScaler()

scaled_X = scaler_x.fit_transform(X_data)
scaled_y = scaler_y.fit_transform(y_data)
np.column_stack([scaled_X, scaled_y])

In [ ]:
X = tf.placeholder(shape=[None, 3], dtype=tf.float32)
y = tf.placeholder(shape=[None, 1], dtype=tf.float32)

W = tf.Variable(tf.random_normal([3, 1]))
b = tf.Variable(tf.random_normal([1]))

H = tf.matmul(X, W) + b

cost = tf.reduce_mean(tf.square(H - y))

train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)

sess = tf.Session()
sess.run(tf.global_variables_initializer())

for step in range(1, 50001):
    _, cost_val = sess.run([train, cost], feed_dict={X:scaled_X, y:scaled_y})
    
    if step%5000==1:
        print(f'{step}번째 cost : {cost_val}')
        
print(f'{step}번째 cost : {cost_val}')

In [ ]:
input_data = np.array([[72, 79, 74],
                       [92, 87, 92]])
# 학습용 Scaler 기준(평균/표준편차 등)으로 변환
scaled_input_data = scaler_x.transform(input_data)
print('scale 조정된 입력값 : \n', scaled_input_data)

# 스케일링된 입력값을 넣고 스케일링된 예측값(hat) 도출
hat = sess.run(H, feed_dict={X:scaled_input_data})

# 스케일링 상태의 예측 결과를 사람이 읽을 수 있는 원본 형식으로 복원
predict_data = scaler_y.inverse_transform(hat)
print('예측 점수 : \n', predict_data)

In [ ]:
input_data = [[90, 85, 90]]
hat = sess.run(H, feed_dict={X:scaler_x.transform(input_data)})
scaler_y.inverse_transform(hat)